# Transformer Architecture: From Scratch to Full Implementation

> **Pure Python / NumPy implementation** — the goal is to understand *how* and *why* each component exists, with no black boxes.

---

## Prerequisites

This notebook is a direct continuation of **`08_attention.ipynb`**, where we studied:
- The Seq2Seq bottleneck and why a fixed context vector fails for long sequences
- Bahdanau additive attention: $e_{t,i} = v^\\top \\tanh(W_a s_t + U_a h_i)$
- Luong multiplicative attention: $e_i = s^\\top h_i$
- **Scaled dot-product attention**: $e_i = \\dfrac{s^\\top h_i}{\\sqrt{d}}$  ← the one the Transformer uses
- The Q/K/V memory analogy: attention as a **differentiable soft lookup**

It also reuses the **embeddings and positional encoding** from `06_embeddings.ipynb`,
**cross-entropy** from `02_loss_functions.ipynb`, and the **`TensorValue` autograd**
from `03_autograd.ipynb`.

The Transformer (Vaswani et al., 2017) took one decisive step further: it **eliminated the RNN entirely**
and applied scaled dot-product attention *directly between all positions in the same sequence*
(**self-attention**), enabling full parallelism during training.

---

## Index

1. [Motivation: N-gram Model](#1-motivation-n-gram-model)
2. [Loading the Data](#2-loading-the-data)
3. [Architecture Overview](#3-architecture-overview)
4. [Scaled Dot-Product Attention](#4-scaled-dot-product-attention)
5. [Multi-Head Attention](#5-multi-head-attention)
6. [Position-wise Feed-Forward Network](#6-position-wise-feed-forward-network)
7. [Layer Normalization](#7-layer-normalization)
8. [Transformer Block](#8-transformer-block)
9. [Complete Model: TransformerLM](#9-complete-model-transformerlm)
10. [Training](#10-training)
11. [Text Generation](#11-text-generation)
12. [Conclusion](#12-conclusion)

> Embeddings and positional encoding are covered in `06_embeddings.ipynb` and are reused here without re-deriving them.

---
## 1. Motivation: N-gram Model

### The Problem: Language Modeling

**Language modeling** is the task of learning the probability distribution over text sequences:

$$P(w_t \mid w_1, w_2, \ldots, w_{t-1})$$

That is: given the previous text, what is the most likely next character (or word)?

Before the Transformer existed, a simple and widely used approach was the **N-gram model**.

### N-gram Model: The Idea

An N-gram model assumes that the next token depends only on the **previous N-1 tokens** (a sliding window).
This is called the **order N-1 Markov assumption**.

```
N=1 (unigram):  P(w_t)                    - ignores context
N=2 (bigram):   P(w_t | w_{t-1})          - looks 1 token back
N=3 (trigram):  P(w_t | w_{t-2}, w_{t-1}) - looks 2 tokens back
```

**Advantages:** Simple, efficient, no iterative training.

**Limitations:**
- Only captures local context (small window)
- Does not generalize well to unseen contexts
- Does not capture long-range dependencies

In [1]:
import sys
import os
import math
import random
from collections import defaultdict, Counter
sys.path.append(os.path.abspath('..'))
sys.path.append(os.path.abspath('..'))
from train import TensorValue
from train import AdamW

# ─── Data loading ───────────────────────────────────────────────────
DATA_PATH = '../database/TinyStories-valid.txt'

with open(DATA_PATH, 'r', encoding='utf-8') as f:
    text_complete= f.read()

# Use the first N characters for speed (pure Python is slower)
MAX_CHARS = 10000000
text = text_complete
# lower case to reduce the number of unique characters
text = text.lower()
words = text.split()

print(f"Total characters in file: {len(text_complete):,}")
print(f"Characters used: {len(text):,}")
print("\nFirst 300 characters:")
print("-" * 50)
print(text[:300])

Total de caracteres no arquivo: 19,432,979
Caracteres utilizados: 19,432,979

Primeiros 300 caracteres:
--------------------------------------------------
 spot. spot saw the shiny car and said, "wow, kitty, your car is so bright and clean!" kitty smiled and replied, "thank you, spot. i polish it every day."
after playing with the car, kitty and spot felt thirsty. they found a small pond with clear water. they drank the water and felt very happy. they


In [2]:
# ─── Model N-gram ────────────────────────────────────────────────────────────

def train_ngram(text,n=3):
    """
    Train a character-level n-gram model.
    
    For each window of (n-1) characters (the context),
    count how often each following character appears.
    
    Returns: dict {context: {next_char: count}}
    """
    model = defaultdict(Counter)
    for i in range(len(text) - n):
        context = tuple(text[i:i + n - 1])  # previous (n-1) characters
        next_char  = text[i + n - 1]            # the character we want to predict
        model[context][next_char] += 1
    return model


def generate_ngram(model, seed, n=3, length=200):
    """
    Generate text by sampling from the n-gram model.
    At each step, take the context of the last (n-1) chars
    and sample the next char proportionally to the counts.
    """
    result = list(seed)
    for _ in range(length):
        context = tuple(result[-(n - 1):])
        if context not in model:
            break
        candidates = model[context]
        total      = sum(candidates.values())
        r          = random.random() * total
        cumulative  = 0
        for char, freq in candidates.items():
            cumulative += freq
            if r <= cumulative:
                result.append(char)
                break
    return ''.join(result)


# Train models with different N
random.seed(42)
seed = text[100:110]  # use a slightly larger seed for richer context

for n in [2, 3, 5, 10]:
    model_ng = train_ngram(text, n=n)
    generated = generate_ngram(model_ng, seed, n=n, length=120)
    print(f"\n── {n}-gram (context of {n-1} char(s)) ──")
    print(repr(generated[:100]))


── 2-gram (contexto de 1 char(s)) ──
'and replier clst, "wisanond henainnghen ha g rd, p she t co andil toutul weshe t my wave therke \nt s'

── 3-gram (contexto de 2 char(s)) ──
'and replies wat ste. stures. she put dayground the th withe gragethe hey. th the ben you’llew saide?'

── 5-gram (contexto de 4 char(s)) ──
'and replied, “hey, why daughed.\nhe rave.\nthe said and always get met in he says, "this hot," max gas'

── 10-gram (contexto de 9 char(s)) ──
'and replied, "i want to try it out. they tried to get it. her mom said that if you can do it by hims'


### Limitations of the N-gram

Observe the results above:

- **Bigram (N=2)**: Almost no coherence - each character depends only on the previous one
- **Trigram (N=3)**: Words start to appear, but the sentences still do not make sense
- **5-gram (N=5)**: Better, but still very local; it repeats memorized patterns

**The fundamental problem:** to understand language, we need to capture long-range dependencies.

For example, in *"The dog that my neighbor bought yesterday __barked__ a lot"*, the verb "barked" must agree with "dog", which appeared much earlier in the text.

The N-gram model cannot do this. **We need something better -> Transformer!**

---
## 2. Loading the Data

### Character-Level Tokenization

To keep things simple, we will use **character-level tokenization** - each unique character is a "token".
The vocabulary is the set of all distinct characters in the text.

Real models use subword tokenization (BPE, WordPiece), but the idea is the same.

In [3]:
# ─── Tokenizer ──────────────────────────────────────────────────────────────
# Context length ()
SEQ_LEN = 32


#Vocab: sorted list of unique characters in the text
chars = sorted(list(set(''.join(text))))
stoi = {s: i for i, s in enumerate(chars)}
itos = {i: s for s, i in stoi.items()}
vocab_size = len(chars)


print(f"Vocab size: {vocab_size}")
print(chars)
print(f"First 40: {repr(''.join(chars[:40]))}")
print(f"Rest:    {repr(''.join(chars[40:]))}")

def encode(texto_input):
    """Converts string to list of integers (token IDs)"""
    return [stoi[ch] for ch in texto_input  if ch in stoi]

def decode(ids):
    """Converts list of IDs back to string"""
    return ''.join(itos.get(i, '?') for i in ids)


def create_batches(tokens, seq_len):
    """
    Creates (input, target) pairs for autoregressive training.
    
    input[i] = tokens[i : i+seq_len]
    target[i] = tokens[i+1 : i+seq_len+1]  ← shifted 1 position
    """
    batches = []
    for i in range(0, len(tokens) - seq_len - 1, seq_len):
        input_seq = tokens[i     : i + seq_len]
        target_seq = tokens[i + 1 : i + seq_len + 1]
        batches.append((input_seq, target_seq))
    return batches


all_tokens  = encode(text)
all_batches = create_batches(all_tokens, SEQ_LEN)
num_batches = len(all_batches)

print(f"\nTotal tokens: {len(all_tokens)}")
print(f"Batches: {num_batches}")
print("\nExample (first 20 tokens):")
print(f"  Input: {repr(decode(all_batches[0][0][:20]))}")
print(f"  Target: {repr(decode(all_batches[0][1][:20]))}")
print("                 ^-- shifted 1 position forward")

Vocab size: 72
['\n', ' ', '!', '"', '#', '$', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '<', '>', '?', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '|', '\xad', '´', 'é', 'ñ', '\u200a', '\u200b', '–', '—', '‘', '’', '“', '”', '…', '🎓']
First 40: '\n !"#$&\'()*+,-./0123456789:;<>?abcdefghi'
Rest:    'jklmnopqrstuvwxyz|\xad´éñ\u200a\u200b–—‘’“”…🎓'

Total tokens: 19432979
Batches: 607280

Example (first 20 tokens):
  Input: ' spot. spot saw the '
  Target: 'spot. spot saw the s'
                 ^-- shifted 1 position forward


---
## 3. Architecture Overview

The Transformer was proposed in the paper *"Attention Is All You Need"* (Vaswani et al., 2017).
The version we will implement is a **decoder-only** model, similar to GPT.

```
                    ┌─────────────────────────────────────-┐
   Input            │          TRANSFORMER LM              │
   tokens ─────────►  Embedding + Pos. Encoding            │
                    │         │                            │
                    │  ┌──────▼──────────────────────┐     │
                    │  │      Transformer Block      │     │
                    │  │  ┌───────────────────────┐  │     │
                    │  │  │ Masked Self-Attention │  │ ×N  │
                    │  │  │   + Add & Layer Norm  │  │     │
                    │  │  └───────────────────────┘  │     │
                    │  │  ┌───────────────────────┐  │     │
                    │  │  │  Feed-Forward Network │  │     │
                    │  │  │   + Add & Layer Norm  │  │     │
                    │  │  └───────────────────────┘  │     │
                    │  └──────┬──────────────────────┘     │
                    │         │                            │
                    │  Linear Projection → Logits          │
                    │         │                            │
                    └─────────┼────────────────────────────┘
                              │
                          Softmax
                              │
                     Next-token probabilities
```

**Data flow:**
1. Integer tokens → dense vectors via **Embedding**
2. Add **Positional Encoding** ("where am I in the sequence?")
3. Pass through N **Transformer Blocks** (attention + FFN)
4. **Linear projection** → logits over the vocabulary
5. **Softmax** → next-token probabilities

---
## 4. Scaled Dot-Product Attention

### The Intuition

The attention mechanism allows each position in the sequence to **"ask about"** and **"receive information from"** any other position.

Analogy: imagine a librarian (Query) looking for books. Each book has a label (Key) and content (Value).
The librarian compares the search with each label and receives a weighted mixture of the contents.

### The Formula

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

- **Q** (Query): "What am I looking for?"
- **K** (Key):   "What does each position have to offer?"
- **V** (Value): "The actual information from each position"
- $\sqrt{d_k}$: scaling factor for numerical stability (without it, the dot product grows with $d_k$, pushing softmax into very small-gradient regions)

### Causal Mask

For text generation, a position **cannot see future positions** (that would be "cheating").
We apply a lower-triangular mask: future positions receive a score of $-\infty$ → weight 0 after softmax.

In [4]:
import numpy as np

def softmax_rows_tensor(x):
    row_max = x.data.max(axis=1, keepdims=True)
    shifted = x - TensorValue(row_max)
    exps = shifted.exp()
    return exps / exps.sum(axis=1, keepdims=True)

def scale_dot_product(Q, K, V, mask=None):
    """
    Scaled dot-product attention.
    Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) V
    Parameters:
        Q: [seq_len × d_k]   — queries
        K: [seq_len × d_k]   — keys
        V: [seq_len × d_v]   — values
        mask: [seq_len × seq_len]  — 0 = ignore, 1 = include
    Returns:
        output:  [seq_len × d_v]
        weights: [seq_len × seq_len]  (for visualization)
    """
    d_k = Q.shape[1]  # keys/queries dimension
    scale = math.sqrt(d_k)

    # Step 1: compatibility scores Q @ K^T
    # scores[i][j] = "how relevant is K[j] to Q[i]?"
    scores = (Q @ K.T())/scale  # [seq × d_k] @ [d_k × seq] -> [seq × seq]
    # divide by sqrt(d_k): numerical stability

    # Step 2: causal mask — prevents position i from seeing position j > i
    if mask is not None:
        mask_array = np.asarray(mask, dtype = bool)
        mask_bias = TensorValue(np.where(mask_array, 0.0, -1e9))  # 0 where mask=1, -1e9 where mask=0
        scores = scores + mask_bias  # add mask bias to scores


    # Step 3: softmax → attention weights (sum to 1 per row)
    weights = softmax_rows_tensor(scores)  # [seq × seq]

    # Step 4: weighted sum of values
    output = weights @ V  # [seq × d_v]

    return output, weights


# ─── Intuitive demonstration ──────────────────────────────────────────────────
print("=" * 60)
print("Demonstration of the Attention Mechanism")
print("=" * 60)
print()
print("Sentence: 'Once upon a time'  (4 tokens, d_k=4)")
print()

random.seed(7)
tokens_demo = list("hello")
seq_a, d_a = len(tokens_demo), 4

Q_a = TensorValue([[random.gauss(0, 0.5) for _ in range(d_a)] for _ in range(seq_a)])
K_a = TensorValue([[random.gauss(0, 0.5) for _ in range(d_a)] for _ in range(seq_a)])
V_a = TensorValue([[random.gauss(0, 0.5) for _ in range(d_a)] for _ in range(seq_a)])

# Causal mask: lower triangular
mask_demo = [[1 if j <= i else 0 for j in range(seq_a)] for i in range(seq_a)]

output_a, weights_a = scale_dot_product(Q_a, K_a, V_a, mask_demo)


print("Causal mask (1=can attend, 0=blocked):")
print("       " + "  ".join(f" {t}" for t in tokens_demo))
for i, row in enumerate(mask_demo):
    print(f"  Q[{tokens_demo[i]}]: " + "  ".join(f" {v}" for v in row))

print()
print("Attention weights (each row sums to 1):")
print("       " + "  ".join(f"{t:>6}" for t in tokens_demo))
for i, row in enumerate(weights_a):
    bar = "  ".join(f"{v.data:.4f}" for v in row)
    print(f"  Q[{tokens_demo[i]}]: {bar}")

print()
print("Observations:")
print("  - Future positions have weight ≈ 0 (mask applied)")
print("  - Each token distributes attention across itself and previous tokens")
print("  - The weights sum to 1.0 in each row")

Demonstration of the Attention Mechanism

Sentence: 'Once upon a time'  (4 tokens, d_k=4)

Causal mask (1=can attend, 0=blocked):
        h   e   l   l   o
  Q[h]:  1   0   0   0   0
  Q[e]:  1   1   0   0   0
  Q[l]:  1   1   1   0   0
  Q[l]:  1   1   1   1   0
  Q[o]:  1   1   1   1   1

Attention weights (each row sums to 1):
            h       e       l       l       o
  Q[h]: 1.0000  0.0000  0.0000  0.0000  0.0000
  Q[e]: 0.5128  0.4872  0.0000  0.0000  0.0000
  Q[l]: 0.3308  0.3342  0.3350  0.0000  0.0000
  Q[l]: 0.2305  0.2595  0.2310  0.2791  0.0000
  Q[o]: 0.1902  0.1715  0.1571  0.2073  0.2739

Observations:
  - Future positions have weight ≈ 0 (mask applied)
  - Each token distributes attention across itself and previous tokens
  - The weights sum to 1.0 in each row


---
## 5. Multi-Head Attention

### Why Multiple Heads?

A single attention head performs a single projection of the representation space. This limits its ability to capture different types of relationships simultaneously.

**Multi-Head Attention** runs `h` attention operations in parallel, each in a subspace:

$$\text{MultiHead}(Q,K,V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h)W^O$$

$$\text{head}_i = \text{Attention}(QW_i^Q,\; KW_i^K,\; VW_i^V)$$

Each head uses different weights ($W_i^Q, W_i^K, W_i^V$), allowing it to learn to focus on different aspects:
- One head may learn syntactic relationships (subject-verb)
- Another may learn coreferences (pronouns and their referents)
- Another may learn positional proximity

### Dimension Split

If `d_model = 32` and `num_heads = 4`, each head operates on `d_head = 32/4 = 8` dimensions.
The computational cost is similar to a single attention mechanism with `d_model` dimensions.

In [5]:
def init_weights(in_dim, out_dim):
    scale = math.sqrt(2.0 / (in_dim + out_dim))
    return TensorValue([[random.gauss(0, scale) for _ in range(out_dim)] for _ in range(in_dim)])
    

class MultiHeadAttention:
    """
    Multi-Head Attention with causal masking.
    d_model: total dimensionality of the model (e.g., 512)
    num_heads: number of attention heads (e.g., 8)
    d_head = d_model // num_heads  # dimensionality per head (e.g., 64)
  
    Learnable parameters:
        W_q, W_k, W_v: [d_model × d_model]  — projections of Q, K, V
        W_o:           [d_model × d_model]  — output projection
    """

    def __init__(self, d_model, num_heads):
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_model   = d_model
        self.num_heads = num_heads
        self.d_head    = d_model // num_heads

        # Projection weights (4 matrices, each [d_model × d_model])
        self.W_q = init_weights(d_model, d_model)
        self.W_k = init_weights(d_model, d_model)
        self.W_v = init_weights(d_model, d_model)
        self.W_o = init_weights(d_model, d_model)

        #parameters for optimization, grouped in a list for easy access wit flat format
        self.parameters = [self.W_q, self.W_k, self.W_v, self.W_o]
        self.total_params = sum(mat.shape[0] * mat.shape[1] for mat in self.parameters)


    def forward(self, x, mask=None):
        """
        x:      [seq_len × d_model]
        Return [seq_len × d_model]
        """
        seq_len = x.shape[0]  # number of tokens in the sequence
        self._x = x
        self._mask = mask

        # Linear projections shared for Q, K, V
        Q_full = x @ self.W_q  # [seq × d_model] x [d_model × d_model] → [seq × d_model]
        K_full = x @ self.W_k  # [seq × d_model] x [d_model × d_model] → [seq × d_model]
        V_full = x @ self.W_v  # [seq × d_model] x [d_model × d_model] → [seq × d_model]

        heads_output    = []
        self._weights_ant = []

        for h in range(self.num_heads):
            ini = h * self.d_head
            end = ini + self.d_head
            # Extract the dimensions for this head → [seq × d_head]
            Q_h = Q_full[:, ini:end]  # [seq × d_model] → [seq × d_head]
            K_h = K_full[:, ini:end]  # [seq × d_model] → [seq × d_head]
            V_h = V_full[:, ini:end]  # [seq × d_model] → [seq × d_head]

            out_h, weights_h = scale_dot_product(Q_h, K_h, V_h, mask)
            heads_output.append(out_h)
            self._weights_ant.append(weights_h)

        # Concatenate heads: [seq × (num_heads * d_head)] = [seq × d_model]
        concat = TensorValue.concat(heads_output, axis=1)  # [seq × d_model]

        # Output projection
        return concat @ self.W_o #[seq × d_model] x [d_model × d_model] → [seq × d_model]




# ─── Demonstration ─────────────────────────────────────────────────────────────
random.seed(42)
D_DEMO, H_DEMO = 8, 2
mha_demo = MultiHeadAttention(D_DEMO, H_DEMO)
x_mha    = TensorValue([[random.gauss(0, 0.3) for _ in range(D_DEMO)] for _ in range(5)])
mac_mha  = [[1 if j <= i else 0 for j in range(5)] for i in range(5)]
out_mha  = mha_demo.forward(x_mha, mac_mha)

print(f"Configuration: d_model={D_DEMO}, num_heads={H_DEMO}, d_head={D_DEMO//H_DEMO}")
print(f"Input:  5 tokens × {D_DEMO} dims")
print(f"Output: {out_mha.shape[0]} tokens × {out_mha.shape[1]} dims")

for h in range(H_DEMO):
    print(f"\nHead {h+1} — attention weights:")
    print("     " + " ".join(f"T{j}  \t" for j in range(5)))
    for i, row in enumerate(mha_demo._weights_ant[h]):
        bar = " ".join(f"{v.data:.3f}\t" for v in row)
        print(f"  T{i}: {bar}")

print("Number of parameters in the model:", mha_demo.total_params)

Configuration: d_model=8, num_heads=2, d_head=4
Input:  5 tokens × 8 dims
Output: 5 tokens × 8 dims

Head 1 — attention weights:
     T0  	 T1  	 T2  	 T3  	 T4  	
  T0: 1.000	 0.000	 0.000	 0.000	 0.000	
  T1: 0.514	 0.486	 0.000	 0.000	 0.000	
  T2: 0.322	 0.335	 0.344	 0.000	 0.000	
  T3: 0.283	 0.250	 0.248	 0.219	 0.000	
  T4: 0.200	 0.197	 0.204	 0.196	 0.203	

Head 2 — attention weights:
     T0  	 T1  	 T2  	 T3  	 T4  	
  T0: 1.000	 0.000	 0.000	 0.000	 0.000	
  T1: 0.422	 0.578	 0.000	 0.000	 0.000	
  T2: 0.372	 0.308	 0.320	 0.000	 0.000	
  T3: 0.205	 0.285	 0.225	 0.285	 0.000	
  T4: 0.199	 0.199	 0.192	 0.209	 0.201	
Number of parameters in the model: 256


---
## 6. Position-wise Feed-Forward Network

### What Is It?

After attention, each position passes through a small neural network **independently**:

$$\text{FFN}(x) = \max(0,\; x W_1 + b_1)\, W_2 + b_2$$

- **Expansion:** `d_model → d_ff` (typically `d_ff = 4 × d_model`)
- **ReLU:** essential nonlinearity (without it, this would just be a linear transformation)
- **Compression:** `d_ff → d_model`

### What Is It For?

Attention mixes information **between positions** (communication). The FFN applies a transformation **within each position** (computation).
This is where the model "processes" and "stores" factual knowledge.

Interestingly, research shows that pairs of neurons in the FFN behave like key-value memories,
associating input patterns with specific outputs.

In [6]:
class FeedForward:
    """
    Feed-Forward network
    FFN(x) = ReLU(x @ W1 + b1) @ W2 + b2
    
    x:    [seq_len × d_model]
    W1:   [d_model × d_ff]      expansion
    W2:   [d_ff × d_model]      compression
    b1/b2: [d_ff] / [d_model]     bias terms
    """

    def __init__(self, d_model, d_ff):
        self.d_model  = d_model # model dimension
        self.d_ff     = d_ff # hidden dimension (expansion)
        self.W1 = init_weights(d_model, d_ff)
        self.b1 = TensorValue([[0.0]*d_ff]) #[1 x d_ff] bias for the hidden layer
        self.W2 = init_weights(d_ff, d_model)
        self.b2 = TensorValue([[0.0]*d_model]) #[1 x d_model] bias for the output layer

        #parameters for optimization
        self.parameters = [self.W1, self.W2, self.b1, self.b2]
        self.total_params = sum(mat.shape[0] * mat.shape[1] for mat in self.parameters)
    def forward(self, x):
        """Forward pass"""
        # Layer 1: linear + bias
        h = x @ self.W1 #[seq × d_model] x [d_model × d_ff] → [seq × d_ff]
        h = h + self.b1

        # ReLU
        h = h.relu()
        self._h_pos = h  # cache post-ReLU

        # Layer 2: linear + bias
        out = h @ self.W2 #[seq × d_ff] x [d_ff × d_model] → [seq × d_model]
        out = out + self.b2

        self._x = x
        return out


# ─── Simulation ─────────────────────────────────────
ff_demo = FeedForward(d_model=8, d_ff=16)
x_ff    = TensorValue([[random.gauss(0, 0.3) for _ in range(8)] for _ in range(4)])
out_ff  = ff_demo.forward(x_ff)

print("Feed-Forward Network:")
print("  Input:  4 tokens × 8 dims")
print("  Hidden (after W1+ReLU): 4 tokens × 16 dims  [expansion 2×]")
print(f"  Output (after W2):          {out_ff.shape[0]} tokens × {out_ff.shape[1]} dims  [back]")

# Show that ReLU sparsifies the representation
h_relu = ff_demo._h_pos
n_zeros = sum(1 for linha in h_relu for v in linha if v == 0.0)
total   = 4 * 16
print(f"\n  Neurons zeroed by ReLU: {n_zeros}/{total} ({100*n_zeros/total:.0f}%)")
print("  → ReLU creates sparsity, which is beneficial for representation")
print("Number of parameters in the model:", len(ff_demo.parameters))



Feed-Forward Network:
  Input:  4 tokens × 8 dims
  Hidden (after W1+ReLU): 4 tokens × 16 dims  [expansion 2×]
  Output (after W2):          4 tokens × 8 dims  [back]

  Neurons zeroed by ReLU: 0/64 (0%)
  → ReLU creates sparsity, which is beneficial for representation
Number of parameters in the model: 4


---
## 7. Layer Normalization

### The Problem with Deep Training

In deep networks, activations can become very large or very small across layers.
This leads to unstable gradients (exploding or vanishing).

### Layer Normalization

For each feature vector $x$ at each position:

$$\hat{x} = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}}$$

$$\text{LayerNorm}(x) = \gamma \odot \hat{x} + \beta$$

- $\mu$, $\sigma^2$: mean and variance computed across the **features** (not the batch)
- $\gamma$, $\beta$: learnable scale and shift parameters
- $\epsilon$: small value for numerical stability

**Difference from Batch Norm:** Layer Norm normalizes across the feature dimension (not the batch),
making it independent of batch size and sequence length.

In [7]:
def layer_norm(x, gamma, beta, eps=1e-5):
    """
    Layer Normalization.
    
    For each token (row of x):
        1. Compute mean and variance across features
        2. Normalize: x_norm = (x - μ) / sqrt(σ² + ε)
        3. Scale and shift: γ * x_norm + β
    
    x:     [seq_len × d_model]
    gamma: [d_model]  — scale (initialized to 1)
    beta:  [d_model]  — shift (initialized to 0)
    """
    mean = x.sum(axis=1, keepdims=True) / x.shape[1]
    x_centered = x - mean
    var = (x_centered * x_centered).sum(axis=1, keepdims=True) / x.shape[1]
    std = (var + eps).pow(0.5)
    x_norm = x_centered / std
    return x_norm * gamma + beta



# ─── Demo ─────────────────────────────────────────────────────────────
print("Effect of Layer Normalization:\n")

x_ln    = [[10.0, 0.1, -5.0, 3.0],
            [0.01, 0.01, 0.01, 0.01],
            [100.0, 200.0, 50.0, 75.0]]
x_ln    = TensorValue(x_ln)
gamma_l = TensorValue([1.0, 1.0, 1.0, 1.0])
beta_l  = TensorValue([0.0, 0.0, 0.0, 0.0])

out_ln  = layer_norm(x_ln, gamma_l, beta_l)

print(f"{'':12} {'d0':>8} {'d1':>8} {'d2':>8} {'d3':>8} | {'mean':>8} {'variance':>10}")
print("-" * 70)

for i in range(x_ln.shape[0]):
    before = x_ln[i]
    after = out_ln[i]
    m_a = before.data.mean()
    v_a = before.data.var()
    m_d = after.data.mean()
    v_d = after.data.var()
    vals_a = " ".join(f"{v:>8.2f}" for v in before.data)
    vals_d = " ".join(f"{v:>8.4f}" for v in after.data)
    print(f"Token {i} before: {vals_a} | μ={m_a:>6.2f}  σ²={v_a:>8.2f}")
    print(f"         after: {vals_d} | μ≈{m_d:>6.4f}  σ²≈{v_d:>6.4f}")
    print()

print("After Layer Norm: mean ≈ 0, variance ≈ 1 for all tokens!")

# Test backward pass of layer normalization
loss_ln = out_ln.sum()
loss_ln.backward()
print("Backward pass completed without errors.")

Effect of Layer Normalization:

                   d0       d1       d2       d3 |     mean   variance
----------------------------------------------------------------------
Token 0 before:    10.00     0.10    -5.00     3.00 | μ=  2.02  σ²=   29.40
         after:   1.4708  -0.3550  -1.2956   0.1798 | μ≈0.0000  σ²≈1.0000

Token 1 before:     0.01     0.01     0.01     0.01 | μ=  0.01  σ²=    0.00
         after:   0.0000   0.0000   0.0000   0.0000 | μ≈0.0000  σ²≈0.0000

Token 2 before:   100.00   200.00    50.00    75.00 | μ=106.25  σ²= 3242.19
         after:  -0.1098   1.6465  -0.9879  -0.5488 | μ≈-0.0000  σ²≈1.0000

After Layer Norm: mean ≈ 0, variance ≈ 1 for all tokens!
Backward pass completed without errors.


---
## 8. Transformer Block

### Putting It All Together: One Block

The **Transformer block** combines all the components we have seen with **residual connections**:

```
x ──┬──── LayerNorm ──► MultiHeadAttn ──► + ─── y1
    └────────────────────────────────────►┘
                                         |
y1 ─┬──── LayerNorm ──► FeedForward ───► + ─── y2
    └────────────────────────────────────►┘
```

**Residual Connections** (skip connections): `output = x + sublayer(x)`
- They allow gradients to flow directly to the earliest layers
- The model learns **corrections** on top of the existing representation, not from scratch
- They are essential for training networks with dozens or hundreds of layers

**Pre-LN vs Post-LN:**
- *Post-LN* (original): `x → sublayer → Add → Norm`
- *Pre-LN* (more stable): `x → Norm → sublayer → Add`  ← we use this here

In [8]:
class TransformerBlock:
    """
     Transformer Block (Pre-LN, decoder-only):
    
    y = x + MultiHeadAttn(LayerNorm(x))
    z = y + FFN(LayerNorm(y))
    """

    def __init__(self, d_model, num_heads, d_ff):
        self.d_model = d_model
        self.attention = MultiHeadAttention(d_model, num_heads)
        self.ff = FeedForward(d_model, d_ff)

        # Layer Norm parameters (γ=1, β=0 by default)
        self.gamma1 = TensorValue([[1.0]*d_model])
        self.beta1 = TensorValue([[0.0]*d_model])
        self.gamma2 = TensorValue([[1.0]*d_model])
        self.beta2 = TensorValue([[0.0]*d_model])

        # list of parameters for optimization
        self.parameters = []
        self.parameters.extend(self.attention.parameters)
        self.parameters.extend(self.ff.parameters)
        self.parameters.append(self.gamma1)
        self.parameters.append(self.beta1)
        self.parameters.append(self.gamma2)
        self.parameters.append(self.beta2)
        self.total_params = sum(mat.shape[0] * mat.shape[1] for mat in self.parameters)

    def forward(self, x, mask=None):
        """
        x: [seq_len × d_model]
        Return: [seq_len × d_model]
        """
        # Sub-layer 1: Self-Attention
        x1_norm   = layer_norm(x, self.gamma1, self.beta1)
        attn_out  = self.attention.forward(x1_norm, mask)
        y         = x +  attn_out      # residual connection

        # Sub-layer 2: Feed-Forward
        y2_norm  = layer_norm(y, self.gamma2, self.beta2)
        ff_out   = self.ff.forward(y2_norm)
        z        = y + ff_out         # residual connection
        return z


# ─── Demo ─────────────────────────────────────────────────────────────
random.seed(0)
bloco = TransformerBlock(d_model=8, num_heads=2, d_ff=16)
x_bl  = TensorValue([[random.gauss(0, 0.3) for _ in range(8)] for _ in range(5)])
mac   = [[1 if j <= i else 0 for j in range(5)] for i in range(5)]
z_bl  = bloco.forward(x_bl, mac)

print("Transformer Block —  dimension check:")
print(f"  Input:  {x_bl.shape[0]} tokens × {x_bl.shape[1]} dims")
print(f"  Output: {z_bl.shape[0]} tokens × {z_bl.shape[1]} dims")
print(f"  Shape preserved? {'Yes' if z_bl.shape[0]==5 and z_bl.shape[1]==8 else 'No'}")

# Verify residual connection: output ≠ input but dimensions are equal
dif = sum(abs(z_bl[0][j].data - x_bl[0][j].data) for j in range(8))
print(f"  Input→Output difference (token 0): {dif:.4f} (should be > 0 due to attention+FFN)")

Transformer Block —  dimension check:
  Input:  5 tokens × 8 dims
  Output: 5 tokens × 8 dims
  Shape preserved? Yes
  Input→Output difference (token 0): 6.9160 (should be > 0 due to attention+FFN)


---
## 9. Complete Model: TransformerLM

Now we stack all the blocks to form the complete language model.

### Final Pipeline

```
token_ids
    ↓
Embedding              tokens → d_model vectors
    ↓
  + Pos. Enc.          adds positional information
    ↓
TransformerBlock × N   attention + FFN (N times)
    ↓
Final LayerNorm        stabilizes the output
    ↓
Linear projection      d_model → vocab_size
    ↓
logits                 score for each token in the vocabulary
    ↓
Softmax                probabilities
```

In [9]:
# We need the positional encoding function for the TransformerLM, so let's implement it here:
class PositionalEncoding:
    """Sinusoidal positional encoding with correct even/odd frequency pairing."""

    def __init__(self, max_len: int, d_model: int):
        self.max_len = max_len
        self.d_model = d_model

        # Standard Transformer PE: even (2k) and odd (2k+1) share the same frequency for each pair k.
        position = np.arange(max_len, dtype=float).reshape(max_len, 1)
        even_dims = np.arange(0, d_model, 2, dtype=float)
        div_term = np.exp(-np.log(10000.0) * (even_dims / d_model))

        pe = np.zeros((max_len, d_model), dtype=float)
        pe[:, 0::2] = np.sin(position * div_term)
        if d_model > 1:
            pe[:, 1::2] = np.cos(position * div_term[: pe[:, 1::2].shape[1]])

        self.positional_encoding = TensorValue(pe)

    def forward(self, positions: list[int]) -> TensorValue:
        """Returns positional-encoding rows for absolute positions."""
        return self.positional_encoding[positions]


class TransformerLM:
    """
    Transformer Language Model (decoder-only, GPT-style).
    Predict next token given previous tokens.
    Predicts P(token_t | token_0, ..., token_{t-1}) for each position.
    """
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_blocks, max_seq_len):
        self.vocab_size  = vocab_size
        self.d_model     = d_model
        self.max_seq_len = max_seq_len

        self.embedding = init_weights(vocab_size, d_model)
        self.pos   = PositionalEncoding(max_seq_len, d_model)  # precomputed
        self.blocks    = [TransformerBlock(d_model, num_heads, d_ff) for _ in range(num_blocks)]

        # Layer norm final
        self.gamma_fin =  TensorValue([[1.0]*d_model])
        self.beta_fin  = TensorValue([[0.0]*d_model])

        # Output projection: d_model → vocab_size
        self.W_out      = init_weights(d_model, vocab_size)

        #parameters for optimization
        self.parameters = []
        self.parameters.append(self.embedding)

        for block in self.blocks:
            self.parameters.extend(block.parameters)
        self.parameters.append(self.gamma_fin)
        self.parameters.append(self.beta_fin)
        self.parameters.append(self.W_out)
        self.total_params = sum(mat.shape[0] * mat.shape[1] for mat in self.parameters)


    def _casual_mask(self, seq_len):
        """Lower-triangular mask: position i can only see j <= i"""
        return [[1 if j <= i else 0 for j in range(seq_len)] for i in range(seq_len)]

    def forward(self, ids):
        """
        ids: list of integers (token IDs)
        Returns: logits [seq_len × vocab_size]
        """
        seq_len = len(ids)

        # 1. Tokens → embeddings
        x =  self.embedding[ids,:] # [seq_len × d_model]

        # 2.  Add positional encoding
        pe = self.pos.positional_encoding[:seq_len, :]  # [seq_len × d_model]
        x  = x + pe

        # 3. Causal mask (decoder)
        mask = self._casual_mask(seq_len)

        # 4. Transformer blocks (attention + FFN, N times)
        for block in self.blocks:
            x = block.forward(x, mask)

        # 5. Final Layer Norm
        x = layer_norm(x, self.gamma_fin, self.beta_fin)
        self._x_final = x

        # 6. Vocab Projection
        logits = x @ self.W_out # [seq_len x d_model] X [d_model x vocab_size] ->   [seq_len × vocab_size]
        return logits
    
    def loss(self,ids_input, ids_target):
        """
        Evaluate the cross-entropy loss for a given (input, target) pair.
        loss = -1/T * Σ_t log P(target_t | context)
        """
        seq_len = len(ids_input)
        logits  = self.forward(ids_input) #[seq_len × vocab_size]

        probs = softmax_rows_tensor(logits)  # [seq_len × vocab_size]
        total_loss = probs[range(seq_len), ids_target].log().sum()  # log P( target_t ) for each position, then sum
    
 
        
        return -total_loss / seq_len

    def generate(self,ids_seed, length=80, temperature=1.0):
        """
        Generate text autoregressively.
        At each step, sample the next token proportional to the probabilities.
        
        temperature > 1 → more uniform distribution (more creative/random)
        temperature < 1 → more concentrated distribution (more deterministic)
        temperature = 0 → argmax (greedy decoding)
        """
        ids = list(ids_seed)
        for _ in range(length):
            # Cut to context window
            ctx    = ids[-self.max_seq_len:]
            logits = self.forward(ctx)
            #select the last token and keep dimensions for softmax
            logits_last = logits[[-1], :]   # last token → next
   
            #implment onlye the greedy decoding for now, we can implement the sampling later
            #greedy temperature = 0
            if temperature == 0:
                next_token =   int(np.argmax(logits_last[0].data))  # argmax → next token ID
            else:
                logits_last = logits_last / temperature  # adjust for temperature
                probs = softmax_rows_tensor(logits_last)  # [1 × vocab_size]
                next_token = int(np.random.choice(self.vocab_size, p=probs.data[0]))  # sample from distribution

            ids.append(next_token)
        return ids
    
    #Test Class
def test_transformer_lm():
    random.seed(42)
    vocab_size = 10
    d_model    = 16
    num_heads  = 4
    d_ff       = 32
    num_blocks = 2
    max_seq_len= 8

    model = TransformerLM(vocab_size, d_model, num_heads, d_ff, num_blocks, max_seq_len)
    optimizer = AdamW(model.parameters, lr=0.001)

    # Test forward pass with dummy input
    input_ids = [0, 1, 2, 3]
    logits = model.forward(input_ids)
    print("Logits shape:", logits.shape[0], "tokens ×", logits.shape[1], "vocab size")
    assert logits.shape[0] == len(input_ids) and logits.shape[1] == vocab_size

    # Test loss computation
    target_ids = [1, 2, 3, 4]  # shifted target
    loss = model.loss(input_ids, target_ids)
    print("Loss:", loss)

    #Test backward pass
    print(model.embedding.grad[0])
    loss.backward()
    print("Backward pass completed without errors.\n")
    print(f"Loss value: {loss.data:.4f}; Grad value:{loss.grad:.4f}\n")
    
    print("Gradients for the first parameter (embedding[0]):")
    print(model.embedding.grad[0])  # should show gradients for the first parameter
    print()
    print("Parameters for the first parameter (embedding[0]):")
    print(model.embedding.data[0])  # should show gradients for the first parameter
    print()
    optimizer.step()  # should update the parameters without errors
    print("Optimizer step completed without errors.\n")
    print("Updated parameters for the first parameter (embedding[0]):")
    print(model.embedding.data[0])  # should show gradients for the first parameter

    # Test generation
    seed_ids = [0, 1]
    generated_ids = model.generate(seed_ids, length=5, temperature=1.0)
    print("Generated IDs:", generated_ids)
    assert len(generated_ids) == len(seed_ids) + 5


test_transformer_lm()

Logits shape: 4 tokens × 10 vocab size
Loss: TensorValue(data=2.718414462397994, grad=0.0, label=)
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
Backward pass completed without errors.

Loss value: 2.7184; Grad value:1.0000

Gradients for the first parameter (embedding[0]):
[ 0.12383658 -0.04670824  0.05374268 -0.45952034 -0.0466494   0.13052629
  0.29313165  0.33888351 -0.01540329 -0.11454679  0.04773601  0.26135936
 -0.06192317 -0.09261221 -0.17862946 -0.23322317]

Parameters for the first parameter (embedding[0]):
[-0.03996347 -0.04795483 -0.03087347  0.19469526 -0.03538662 -0.41529112
  0.09216853 -0.07414608 -0.06017351  0.03214066  0.0644278   0.32271312
  0.1821182   0.03064918 -0.20477357 -0.28141671]

Optimizer step completed without errors.

Updated parameters for the first parameter (embedding[0]):
[-0.04096343 -0.04695478 -0.03187343  0.19569506 -0.03438659 -0.4162907
  0.09116843 -0.075146   -0.05917345  0.03314062  0.06342774  0.32171279
  0.18311802  0.03164915 -0.20

---
## 10. Training

### Cross-Entropy Loss

To train the language model, we use **cross-entropy** between the predicted distribution and the correct token:

$$\mathcal{L} = -\frac{1}{T} \sum_{t=1}^{T} \log P(\text{token}_t \mid \text{token}_1, \ldots, \text{token}_{t-1})$$

**Perplexity** is a more intuitive metric:

$$\text{Perplexity} = e^{\mathcal{L}}$$

- A model that guesses all tokens equally has perplexity = `vocab_size`
- The lower the perplexity, the better

### Backpropagation

The cross-entropy gradient with respect to the logits has an elegant form:

$$\frac{\partial \mathcal{L}}{\partial \text{logit}_j} = p_j - \mathbb{1}[j = \text{target}]$$

In other words: subtract 1 from the correct token and leave the rest as the predicted probabilities. That gradient then flows backward through the network.

In [10]:
# ─── Hyperparameters (kept small to run in pure Python) ─────────────────────
# Note: in production, use NumPy/PyTorch with larger models.
# These values are intentionally small for didactic purposes.

D_MODEL    = 32      # embedding dimension
NUM_HEADS  = 4       # attention heads
D_FF       = 32      # FFN inner dimension
NUM_BLOCKS = 2       # number of transformer blocks
MAX_CTX    = SEQ_LEN # maximum context length
LR         = 0.005   # learning rate
N_TRAIN   = 10000      # training samples
N_EPOCHS   = 50       # epochs
VOCAB_SIZE = vocab_size #vocab size (number of unique tokens)

print("=" * 55)
print("   MODEL CONFIGURATION")
print("=" * 55)
print(f"  Vocabulary:          {VOCAB_SIZE} tokens")
print(f"  d_model:             {D_MODEL}")
print(f"  Attention heads:     {NUM_HEADS}")
print(f"  d_ff:                {D_FF}")
print(f"  Transformer blocks:  {NUM_BLOCKS}")
print(f"  Context length:      {MAX_CTX}")
print(f"  Learning rate:       {LR}")
print(f"  Training samples:    {N_TRAIN}")
print(f"  Epochs:              {N_EPOCHS}")

random.seed(42)
model = TransformerLM(
    vocab_size  = VOCAB_SIZE,
    d_model     = D_MODEL,
    num_heads   = NUM_HEADS,
    d_ff        = D_FF,
    num_blocks  = NUM_BLOCKS,
    max_seq_len = MAX_CTX
)

n_params = model.total_params
print(f"\n  Total parameters: ~{n_params:,}")
print(f"  Baseline loss (rand): {math.log(VOCAB_SIZE):.4f}")
print(f"  Baseline perplexity: {VOCAB_SIZE} (entire vocabulary)")

   MODEL CONFIGURATION
  Vocabulary:          72 tokens
  d_model:             32
  Attention heads:     4
  d_ff:                32
  Transformer blocks:  2
  Context length:      32
  Learning rate:       0.005
  Training samples:    10000
  Epochs:              50

  Total parameters: ~17,344
  Baseline loss (rand): 4.2767
  Baseline perplexity: 72 (entire vocabulary)


In [11]:
import time

# ─── Prepare training data ────────────────────────────────────────────────────
batches_train = all_batches[:N_TRAIN]

# ─── Training loop ────────────────────────────────────────────────────────────
historic_loss = []
t_start = time.time()

# Optimizer:
optimizer = AdamW(model.parameters, LR)

rand_index = random.randint(0, len(text)-SEQ_LEN)
seed_text = text[rand_index:rand_index+SEQ_LEN]
seed_ids  = encode(seed_text)
print(f"Seed: {repr(seed_text)}\n")

print("Starting training...")
print(f"{'Epoch':>6} | {'Loss':>8} | {'Perplexity':>14} | {'Time (s)':>10}")
print("-" * 48)



for epoch in range(N_EPOCHS):
    random.shuffle(batches_train)
    total_loss = 0.0

    for input, target in batches_train:
        loss = model.loss(input, target)
        #print(f"  Sample loss: {loss.data:.4f}  (input: {decode(input)}, target: {decode(target)})")
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.data

    loss_media   = total_loss / len(batches_train)
    perplexidade = math.exp(min(loss_media, 20))
    historic_loss.append(loss_media)

    t_elapsed = time.time() - t_start
    print(f"{epoch+1:>6} | {loss_media:>8.4f} | {perplexidade:>14.2f} | {t_elapsed:>10.1f}")
    ids_g = model.generate(seed_ids, length=120, temperature=0)
    generated_text = decode(ids_g)
    print(f"Generated text:\n{generated_text}\n")

print()
print(f"Training completed in {time.time()-t_start:.1f}s")

# ─── Loss curve ───────────────────────────────────────────────────────────────
print("\nLoss curve during training:")
loss_min = min(historic_loss)
loss_max = max(historic_loss)
amp      = max(loss_max - loss_min, 0.01)

for i, l in enumerate(historic_loss):
    barras  = int(40 * (l - loss_min) / amp)
    barra_s = '█' * barras + '░' * (40 - barras)
    print(f"  Epoch {i+1}: {barra_s} {l:.4f}")

reducao = (historic_loss[0] - historic_loss[-1]) / historic_loss[0] * 100
print(f"\nLoss reduction: {reducao:.1f}%")

Starting training...
 Epoch |     Loss |     Perplexity |   Time (s)
------------------------------------------------
Seed: 'nce upon a time there was a lady'

     1 |   1.9948 |           7.35 |       39.4
Generated text:
nce upon a time there was a lady to said the sped the sped the sped the sped the sped the sped the sped the sped the sped the sped the sped the sped the

     2 |   1.7144 |           5.55 |       80.2
Generated text:
nce upon a time there was a lady to to the the the the the the the the the the the the the the the the the the the the the the the the the the the the t

     3 |   1.6387 |           5.15 |      119.5
Generated text:
nce upon a time there was a lady to partiners and the bound to partiners and the bound to partiners and the bound to partiners and the bound to partiner

     4 |   1.5968 |           4.94 |      158.7
Generated text:
nce upon a time there was a lady and saw the saw the saw the saw the saw the saw the saw the saw the saw the saw the saw t

---
## 11. Text Generation

### Autoregressive Decoding

To generate text, the model runs in a loop:

```
seed → [model] → logits → softmax → sample → next token
   ↑                                                  |
   └────────────── append to the sequence ────────────┘
```

**Temperature** controls diversity:
- `T → 0`: always chooses the most likely token (greedy, repetitive)
- `T = 1`: samples from the learned distribution
- `T > 1`: more random, more creative (but less coherent)

In [14]:
# ─── Text generation ──────────────────────────────────────────────────────────
print("=" * 60)
print("TEXT GENERATION")
print("=" * 60)
rand_index = random.randint(0, len(text)-SEQ_LEN)
seed_text = text[rand_index:rand_index+SEQ_LEN]
seed_ids  = encode(seed_text)
print(f"Seed: {repr(seed_text)}\n")

for temp in [0.0, 0.7, 1.2]:
    random.seed(42)
    ids_g = model.generate(seed_ids, length=120, temperature=temp)
    generated_text = decode(ids_g)
    temp_name = "greedy" if temp == 0.0 else f"T={temp}"
    print(f"── {temp_name} ──")
    print(f"  {repr(generated_text[:100])}")
    print()

TEXT GENERATION
Seed: "ew helicopter, so it wouldn't be"

── greedy ──
  "ew helicopter, so it wouldn't be a be a be a be a be a be a be a be a be a be a be a be a be a be a "

── T=0.7 ──
  "ew helicopter, so it wouldn't be stle cawa a big said and her lily and coun was very gary bigh saw a"

── T=1.2 ──
  "ew helicopter, so it wouldn't be,e cabe coused. benned sam con, treis on. hnow lone startacll, lily "



In [15]:
# ─── Comparison: N-gram vs Transformer ────────────────────────────────────────
print("=" * 60)
print("FINAL COMPARISON: N-gram (3) vs Transformer")
print("=" * 60)
print(f"Seed: {repr(seed_text)}\n")

modelo_3g = train_ngram(text, n=3)

random.seed(42)
ng_txt = generate_ngram(modelo_3g, seed_text, n=3, length=120)

random.seed(42)
ids_tf = model.generate(seed_ids, length=120, temperature=1.0)
tf_txt = decode(ids_tf)

print("3-gram:")
print(f"  {repr(ng_txt[:100])}")
print()
print("Transformer:")
print(f"  {repr(tf_txt[:100])}")
print()

# ─── Sequence probability (log-likelihood) ────────────────────────────────────
print("-" * 60)
print("Log-likelihood on a test sequence:")
ids_test = all_batches[-1][0]  # sequence not seen during training
ids_target  = all_batches[-1][1]

logits_tf = model.forward(ids_test)
ll = 0.0
probs = softmax_rows_tensor(logits_tf)  # [seq_len × vocab_size]
p  = probs[range(len(ids_target)), ids_target]  # P(target_t) for each position
ll = p.log().sum().data  # log-likelihood of the sequence
print(f"  Transformer  — log-likelihood: {ll:.2f}  (loss: {-ll/len(ids_target):.4f})")

print()
print("Note: with longer training and a larger model, the Transformer")
print("greatly surpasses the N-gram in coherence and generalization!")

FINAL COMPARISON: N-gram (3) vs Transformer
Seed: "ew helicopter, so it wouldn't be"

3-gram:
  "ew helicopter, so it wouldn't begs. hend said, ands. hine say, mod to frock the the whomethee. her a"

Transformer:
  "ew helicopter, so it wouldn't beicke no spidvecart to look out out a putun in a bos tall and are of "

------------------------------------------------------------
Log-likelihood on a test sequence:
  Transformer  — log-likelihood: -56.79  (loss: 1.7746)

Note: with longer training and a larger model, the Transformer
greatly surpasses the N-gram in coherence and generalization!


---
## 12. Conclusion

### What We Built

We implemented a complete **decoder-only Transformer** in pure Python, including:

| Component | Function |
|---|---|
| **Embedding** | Represents tokens as dense vectors |
| **Positional Encoding** | Tells the model the order of tokens |
| **Scaled Attention** | Measures Query-Key compatibility and weights Values |
| **Multi-Head Attention** | H parallel attentions to capture varied patterns |
| **Feed-Forward** | Nonlinear transformation at each position |
| **Layer Normalization** | Stabilizes activations and makes training easier |
| **Residual Connections** | Let gradients flow freely, enabling deep networks |
| **Cross-Entropy + Backprop** | End-to-end supervised training |

### Why the Transformer Dominates

1. **Global Attention:** each token can relate to any other in O(1) layers (vs. O(n) in RNNs)
2. **Parallelism:** unlike RNNs, all tokens are processed simultaneously during training
3. **Scalability:** you can just stack more blocks and increase `d_model` for more powerful models
4. **Flexibility:** the same architecture works for text, images, audio, code, and more

### Next Steps

- **GPT:** decoder only, more layers, BPE tokenization
- **BERT:** bidirectional encoder, masked language modeling
- **T5/BART:** encoder + decoder for seq2seq
- **Flash Attention:** memory-efficient attention for long contexts
- **Rotary Embeddings (RoPE):** modern relative positional encoding

### References

- Vaswani et al. (2017), *Attention Is All You Need*
- Radford et al. (2019), *Language Models are Unsupervised Multitask Learners* (GPT-2)
- Andrej Karpathy - *nanoGPT* (clean PyTorch implementation)
- MultiHead: https://theaisummer.com/self-attention/
- Self attention explanation: 
    - https://medium.com/@manindersingh120996/the-detailed-explanation-of-self-attention-in-simple-words-dec917f83ef3
    - https://sebastianraschka.com/blog/2023/self-attention-from-scratch.html

---

> **Performance Note:** This implementation uses pure Python for educational purposes.
> In production, use **PyTorch** or **JAX** - they are 1000x faster thanks to optimized
> matrix operations on GPU/CPU (BLAS, CUDA, cuDNN).